# Canadian case NTGS1 / NTGS2: fidelity porting against GEOtop 3.0

The snow+soil column of WHETGEO-1D in the **fidelity** configuration is run **heat only** (`soilWaterMode = Fixed`, as GEOtop's `WaterBalance = 0`). It is compared with **GEOtop 3.0 without canopy, corrected** (`casi/Canada_NTGS/sims/<case>_senza_chioma_corretta`) on four cases: two sites (NTGS1 peatland, NTGS2 hilltop) × two reanalyses (ERA5, JRA-3Q).

**Run setup**

- Hourly steps from 01/09/2020; comparison over **01/09/2021 – 01/09/2023**, after one year of warm-up as in GEOtop's project.
- Same 20 soil layers to 9.03 m, same initial temperature per layer, zero heat flux at the bottom.

**Forcing**

The forcing is GEOtop's own, recomputed from its input files by the temporary fidelity tools; it matches GEOtop's point output to 1e-6:

- precipitation × cos(slope) once, split at 1 / −3 °C;
- shortwave on the slope;
- the reanalysis longwave;
- pressure from the elevation.

**Fallback.** Without snow, the top of the soil is held at GEOtop's `Tsurface` (Dirichlet): the bare-soil energy balance now exists, but here GEOtop's surface temperature keeps the comparison on the snow and the soil.

**Ported for fidelity** (rounds 4–5): GEOtop's skin node (SURF-1/2), shortwave penetration (SNOW-7), thin-snow regime, snow–soil face conductance (SNOW-3), porosity cap (SNOW-6) and thin cells kept unshrunk. **Not ported:** delayed refreezing (STA-3) and GEOtop's emergency turbulence path, which never acts in these cases.

Nothing was tuned.

Run `cases/snowsoil/canada_ntgs/run.sh` first. This notebook reads `output/<case>/` and `geotop/<case>/reference.csv.gz`.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
CASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "cases", "snowsoil", "canada_ntgs"))
sys.path.insert(0, CASE_DIR)
import compare
pd.set_option("display.width", 200)
data = {c: compare.load(c) for c in compare.CASES}

## Comparison table

For scalar rows, the columns hold the porting−GEOtop bias and the RMSE. For peaks and maximum thaw they hold porting / GEOtop. For snow dates they hold porting − GEOtop [d] / GEOtop's date.

In [ ]:
tables = {c: compare.compare(c).set_index(["quantity", "unit"]) for c in compare.CASES}
summary = pd.concat({c: t for c, t in tables.items()}, axis=1)
summary.round(2)

## Snow depth and SWE (daily means)

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 13), sharex=True)
for i, c in enumerate(compare.CASES):
    d = data[c].resample("D").mean()
    for j, q in enumerate(("HS", "SWE")):
        ax = axes[i, j]
        ax.plot(d.index, d["gt_" + q], "k", label="GEOtop 3.0")
        ax.plot(d.index, d["wg_" + q], "tab:blue", label="porting (fidelity)")
        ax.set_ylabel(q + " [mm]")
        ax.set_title(c)
axes[0, 0].legend()
fig.tight_layout()

## Soil temperature at 0.1, 0.5, 1, 2 and 4 m (daily means)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 15), sharex=True)
colors = dict(zip(compare.DEPTHS_MM, ["tab:red", "tab:orange", "tab:green", "tab:blue", "tab:purple"]))
for ax, c in zip(axes, compare.CASES):
    d = data[c].resample("D").mean()
    for z in compare.DEPTHS_MM:
        ax.plot(d.index, d["gt_T%d" % z], color=colors[z], lw=1, label="%.1f m GEOtop" % (z / 1000))
        ax.plot(d.index, d["wg_T%d" % z], color=colors[z], lw=1, ls="--", label="%.1f m porting" % (z / 1000))
    ax.axhline(0, color="grey", lw=0.5)
    ax.set_ylabel("T [°C]")
    ax.set_title(c)
axes[0].legend(ncol=5, fontsize=8)
fig.tight_layout()

## Monthly bias of soil temperature, porting − GEOtop

The bias builds up in winter, under snow, and it is largest in spring at shallow depths. In summer, when the top is held at GEOtop's `Tsurface`, the shallow layers return to GEOtop's values; the deep layers keep part of the winter deficit.

In [ ]:
rows = {}
for c in compare.CASES:
    m = data[c].resample("MS").mean()
    rows[c] = pd.DataFrame({"%.1f m" % (z / 1000): m["wg_T%d" % z] - m["gt_T%d" % z] for z in compare.DEPTHS_MM})
pd.concat(rows, axis=1).round(2)

## Surface temperature with snow (hourly, both models with snow)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, c in zip(axes, compare.CASES):
    d = data[c]
    both = (d.gt_SWE > compare.SNOW_SWE) & (d.wg_SWE > compare.SNOW_SWE)
    ax.plot(d.loc[both, "gt_Ts"], d.loc[both, "wg_Ts"], ".", ms=1, alpha=0.3)
    ax.plot([-50, 5], [-50, 5], "k", lw=0.5)
    ax.set_xlabel("GEOtop Ts [°C]"); ax.set_ylabel("porting Ts [°C]"); ax.set_title(c)
fig.tight_layout()

## Thaw depth (GEOtop's `lowest_thawed_soil_depth` and the same algorithm on the porting's profiles)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
for ax, c in zip(axes, compare.CASES):
    d = data[c].resample("D").mean()
    ax.plot(d.index, d.gt_thaw, "k", label="GEOtop 3.0")
    ax.plot(d.index, d.wg_thaw, "tab:blue", label="porting")
    ax.invert_yaxis(); ax.set_ylabel("thaw depth [mm]"); ax.set_title(c)
axes[0].legend()
fig.tight_layout()

## Attribution of the differences

With the skin node and the shortwave penetration the porting follows GEOtop:

- **Snow:** SWE daily RMSE 0.4–1.0 mm, peaks within 0.5–2 %, snow-on and snow-off on the same day (one day off in two of the sixteen dates).
- **Surface with snow:** bias within ±0.02 K, RMSE 0.4–0.5 K.
- **Soil:** bias +0.02 to +0.13 K, daily RMSE 0.23–0.35 K at 0.1–1 m and 0.10–0.13 K at 4 m. What is left: 0.5–0.8 K warmer at 0.5–1 m in January–February, up to 0.35 K colder in the top 0.1 m in November–December.
- **Thaw depth:** the summer maximum is 16–26 mm deeper than GEOtop's.

**How the 1–1.6 K cold soil of the first runs was closed** (soil bias at 0.1 / 1 / 4 m, NTGS1 ERA5):

| porting | soil bias [K] | SWE daily RMSE | snow-off |
|---|---|---|---|
| first runs | −1.42 / −1.18 / −1.02 | 5.7 mm | +3, +5 d |
| + GEOtop's face conductance, thin snow, porosity cap | −1.58 / −1.33 / −1.15 | 5.8 mm | +3, +4 d |
| + skin node | −0.51 / −0.35 / −0.31 | 4.0 mm | +2, +2 d |
| + shortwave penetration | +0.04 / +0.13 / +0.06 | 0.38 mm | 0, 0 d |

1. **Skin node.** An instrumented GEOtop 3.0 (`INDAGINE_SUOLO.md` in the case folder of the GEO2GEO workspace) showed that GEOtop without its skin is 1.07–1.32 K colder than with it: the surface flux reaches the centre of the top layer through half of its thickness, and without that resistance the pack insulates less. GEOtop freezes the resistances at the top-layer temperature at the start of the step, which is where the negative Obukhov lengths of 12–45 % of the snow hours come from; the porting does the same.
2. **Shortwave penetration.** With the skin, the remaining deficit grew from February to April with the sun: in December–January the porting's H matched GEOtop's, in February–April it was 1.6–3.6 W m⁻² smaller. With more than one snow layer GEOtop gives no shortwave to the skin (`rad_snow_absorption`, `radiation.cc:582-613`): the top layer absorbs it and starts the next step warmer, and the colder skin draws more heat from the air.
3. **Not the emergency turbulence path.** The recomputation without stable correction after 10 Newton iterations below −50 °C (`energy.balance.cc:57-58, 1835, 2618-2621`) never fires in these cases (0 of about 17 000 snow steps; switched off, GEOtop's outputs are byte-identical).
4. **Not the conductivities within the step.** Refreshing them three times per step (`-DheatPicardIterations=3`) moves the soil by less than 0.002 K.

**Numerics.**

- About 65–95 s per case: 3 years hourly, 20 soil cells, 26 280 steps.
- No step is rejected or degraded. The few steps that used to be rejected and halved had the surface sweeps alternating across 0 °C, where the balance over snow jumps between sublimation and evaporation; the solver treats them as a skin at 0 °C.
- In NTGS1 JRA-3Q, one hour of 5 October 2021 (a saturated peat cell forming its first ice exactly at its melting temperature) needs a second polish round of the heat driver: Newton converges only linearly there (743 → 9.4 → 0.005 times the per-cell bound).
